# Lab 4: Prompt Playground & Prompt Hub

## Difficulty: Beginner | ~35 min | Requires Lab 1

Learn how to version, store, and retrieve prompts using LangSmith's Prompt Hub — the Git for prompts.

### What is the Prompt Playground?

The Prompt Playground is a web UI in LangSmith (at smith.langchain.com) where you can test prompts against different models, adjust parameters like temperature and max tokens, and try few-shot examples — all without writing code. It's the fastest way to iterate on a prompt before committing it to your codebase.

### What is Prompt Hub?

Prompt Hub is a version-controlled registry for prompts. Every time you push a prompt, it creates a new immutable version. You can pull any version by name or commit hash, share prompts across your team, and tag versions for environment management (dev, staging, production).

### What are Few-Shot Prompts?

A few-shot prompt includes example input/output pairs before the actual question. This teaches the model the exact format and behavior you want, without fine-tuning. For example, showing two tool-selection examples helps the model learn the pattern on the third query.

In [ ]:
!pip install -qU langsmith>=0.1.0 langchain-core>=0.2.0 openai>=1.0.0 python-dotenv>=1.0.0

This installs the exact versions of every library used in this lab.

## Cell 2: Load Environment and Initialize Clients

Loads API keys and initializes both the LangSmith client (for Hub operations) and the OpenAI client (for LLM calls).

In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client
from openai import OpenAI

# Load API keys from .env file
load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY"

# LangSmith client for Hub operations (push/pull/list prompts)
ls_client = Client()

# OpenAI client pointing to OpenRouter for free LLM calls
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
print("✓ Clients ready")

## Cell 3: Create a System Prompt and Push It to Hub

We create a `ChatPromptTemplate` for tool selection and push it to Prompt Hub. This creates version 1 of the prompt.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a system prompt that routes user queries to the correct tool
system_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that routes user queries to the correct tool. Available tools: calculator, search, summarize. Analyze the query and decide which tool to use."),
    ("user", "{query}")
])

# Push the prompt to Prompt Hub — this creates version 1
url = ls_client.push_prompt("tool-selector", object=system_prompt)
print(f"✓ Pushed to Hub: {url}")

The `push_prompt()` method stores the prompt in your workspace and returns a URL where you can view it in the LangSmith UI. You can also open the Playground from that URL to test the prompt interactively.

## Cell 4: List Prompts in Your Workspace

Query Prompt Hub to see all stored prompts — a quick way to verify your push worked.

In [ ]:
prompts = list(ls_client.list_prompts(limit=10))
print(f"✓ Found {len(prompts)} prompt(s):")
for p in prompts:
    print(f"  - {p.repo_handle}")

You should see `tool-selector` in the list. The `repo_handle` is the unique name you use to pull the prompt later.

## Cell 5: Pull the Prompt by Name

Pull the latest version of your prompt from Hub. The returned object is a `ChatPromptTemplate` you can invoke with variables.

In [ ]:
pulled = ls_client.pull_prompt("tool-selector")
print(f"✓ Pulled prompt: {type(pulled).__name__}")
print(f"  Messages: {len(pulled.messages)}")
for msg in pulled.messages:
    content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
    print(f"  {msg.type}: {content}")

The pulled prompt is the exact `ChatPromptTemplate` you pushed — round-trip fidelity. You can now use it anywhere in your codebase without hardcoding the prompt text.

## Cell 6: Invoke the Prompt with a Variable

Test the pulled prompt by filling in the `{query}` variable with a sample question.

In [ ]:
formatted = pulled.invoke({"query": "What is 15 multiplied by 23?"})
print("✓ Formatted messages:")
for msg in formatted.messages:
    print(f"  {msg.type}: {msg.content}")

The `invoke()` method fills in the `{query}` variable and returns formatted messages ready to send to an LLM.

## Cell 7: Test the Prompt with an LLM

Send the formatted prompt to the LLM via OpenRouter and print the response.

In [ ]:
response = openai_client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[{"role": msg.type, "content": msg.content} for msg in formatted.messages]
)
print(f"✓ LLM Response: {response.choices[0].message.content}")

The model should identify that this query needs the calculator tool. Check the LangSmith UI to see the trace of this LLM call.

## Cell 8: Refine the Prompt and Push Version 2

Improve the prompt to produce more structured output, then push it as version 2. The prompt name stays the same — LangSmith versions it automatically.

In [ ]:
refined_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a tool-routing assistant. Given a user query, respond with ONLY the tool name (calculator, search, or summarize) and a brief reason. Format: Tool: [name] | Reason: [why]"),
    ("user", "{query}")
])

# Push as version 2 — same name, new version
url = ls_client.push_prompt("tool-selector", object=refined_prompt)
print(f"✓ Pushed refined version: {url}")

Every push to the same prompt name creates a new version. Version 1 is preserved — you can always pull it back.

## Cell 9: Pull by Specific Version and Compare

Pull both versions side by side to see how the prompt evolved. Pulling with `:1` or `:2` pins to a specific version.

In [ ]:
v1 = ls_client.pull_prompt("tool-selector:1")
v2 = ls_client.pull_prompt("tool-selector:2")

print("✓ Version 1 system message:")
print(f"  {v1.messages[0].content[:120]}...")
print(f"\n✓ Version 2 system message:")
print(f"  {v2.messages[0].content[:120]}...")

Version pinning (`pull_prompt("name:2")`) ensures your code doesn't break when someone pushes a new version. Use this in production code.

## Cell 10: Test the Refined Prompt

Run the refined prompt with a different query to see the structured output format in action.

In [ ]:
formatted_v2 = v2.invoke({"query": "Summarize the latest news about AI"})
response = openai_client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[{"role": msg.type, "content": msg.content} for msg in formatted_v2.messages]
)
print(f"✓ Refined prompt response:")
print(f"  {response.choices[0].message.content}")

The refined prompt should produce a clearer, more structured response with the tool name and reason — easier for downstream code to parse.